# 00 - Setup Lakehouse

## Purpose

Prepare the shared configuration used by the Retail Banking Customer Analytics blueprint. This notebook establishes the source path, table naming conventions, batch identifiers, helper functions, and source file validation used throughout the project.

## Concepts covered

- Fabric Lakehouse file and table layout
- Source file conventions under Files/retail_banking/source
- Bronze, Silver, and Gold naming standards
- Batch IDs and run timestamps
- Row count logging helpers
- Safe, repeatable setup for beginner demos

## Prerequisites

- A Microsoft Fabric Lakehouse attached to this notebook.
- Sample CSV files uploaded to Files/retail_banking/source.
- Permission to create or overwrite Lakehouse Delta tables.

## Expected output

A run configuration summary, optional folder creation, source file readiness results, and reusable helper functions.

In [ ]:
from datetime import datetime, timezone
from typing import Dict, Iterable

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DecimalType, StringType, StructField, StructType, TimestampType

PROJECT_NAME = "fabric-data-engineering-blueprint"
BUSINESS_DOMAIN = "retail_banking"
ENVIRONMENT = "dev"
BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

SOURCE_BASE_PATH = "Files/retail_banking/source"
CONFIG_BASE_PATH = "Files/retail_banking/config"
CHECKPOINT_BASE_PATH = "Files/retail_banking/checkpoints"
REPORT_BASE_PATH = "Files/retail_banking/reports"

ENTITIES = ["customers", "accounts", "products", "branches", "transactions"]
GOLD_TABLES = ["dim_customer", "dim_account", "dim_product", "dim_branch", "dim_date", "fact_transaction"]

def log_info(message: str) -> None:
    print(f"[INFO] {datetime.now(timezone.utc).isoformat()} | {message}")

def log_warning(message: str) -> None:
    print(f"[WARN] {datetime.now(timezone.utc).isoformat()} | {message}")

log_info(f"Project: {PROJECT_NAME}")
log_info(f"Business domain: {BUSINESS_DOMAIN}")
log_info(f"Environment: {ENVIRONMENT}")
log_info(f"Batch ID: {BATCH_ID}")
log_info(f"Source path: {SOURCE_BASE_PATH}")

## Lakehouse folder layout

In Fabric, Spark can read Lakehouse files using the Files/... path. Delta tables created with saveAsTable are stored in the Lakehouse Tables area.

Recommended layout:

- Files/retail_banking/source for CSV source files
- Files/retail_banking/config for optional YAML and Python configuration
- Files/retail_banking/reports for generated reports
- Lakehouse Tables for Bronze, Silver, and Gold Delta tables

In [ ]:
def get_fabric_file_utils():
    candidates = []
    try:
        candidates.append(notebookutils)  # Provided by Fabric notebooks when available.
    except NameError:
        pass
    try:
        candidates.append(mssparkutils)  # Legacy Synapse name, still seen in some environments.
    except NameError:
        pass

    for candidate in candidates:
        if hasattr(candidate, "fs"):
            return candidate.fs
    return None

def ensure_folder(path: str) -> None:
    file_utils = get_fabric_file_utils()
    if file_utils is None:
        log_warning(f"Fabric file utilities are not available. Skipping folder creation for {path}.")
        return
    try:
        file_utils.mkdirs(path)
        log_info(f"Folder is ready: {path}")
    except Exception as exc:
        log_warning(f"Could not create or validate folder {path}: {exc}")

for folder in [SOURCE_BASE_PATH, CONFIG_BASE_PATH, CHECKPOINT_BASE_PATH, REPORT_BASE_PATH]:
    ensure_folder(folder)

## Source contracts

Bronze ingestion reads source values as strings so the raw structure is preserved. The schema dictionary below documents the expected business fields and can be reused for stricter production validation.

In [ ]:
SOURCE_SCHEMAS: Dict[str, StructType] = {
    "customers": StructType([
        StructField("customer_id", StringType(), True), StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True), StructField("email", StringType(), True),
        StructField("phone", StringType(), True), StructField("city", StringType(), True),
        StructField("state", StringType(), True), StructField("country", StringType(), True),
        StructField("customer_segment", StringType(), True), StructField("customer_status", StringType(), True),
        StructField("date_of_birth", DateType(), True), StructField("created_date", DateType(), True),
    ]),
    "accounts": StructType([
        StructField("account_id", StringType(), True), StructField("customer_id", StringType(), True),
        StructField("product_id", StringType(), True), StructField("account_type", StringType(), True),
        StructField("account_status", StringType(), True), StructField("opened_date", DateType(), True),
        StructField("closed_date", DateType(), True), StructField("current_balance", DecimalType(18, 2), True),
        StructField("branch_id", StringType(), True),
    ]),
    "products": StructType([
        StructField("product_id", StringType(), True), StructField("product_name", StringType(), True),
        StructField("product_category", StringType(), True), StructField("product_family", StringType(), True),
        StructField("fee_model", StringType(), True), StructField("is_active", StringType(), True),
        StructField("launch_date", DateType(), True),
    ]),
    "branches": StructType([
        StructField("branch_id", StringType(), True), StructField("branch_name", StringType(), True),
        StructField("city", StringType(), True), StructField("state", StringType(), True),
        StructField("region", StringType(), True), StructField("opened_date", DateType(), True),
        StructField("is_active", StringType(), True),
    ]),
    "transactions": StructType([
        StructField("transaction_id", StringType(), True), StructField("account_id", StringType(), True),
        StructField("product_id", StringType(), True), StructField("branch_id", StringType(), True),
        StructField("transaction_timestamp", TimestampType(), True), StructField("transaction_type", StringType(), True),
        StructField("transaction_channel", StringType(), True), StructField("transaction_amount", DecimalType(18, 2), True),
        StructField("currency", StringType(), True), StructField("transaction_status", StringType(), True),
    ]),
}

def source_file(entity_name: str) -> str:
    return f"{SOURCE_BASE_PATH}/{entity_name}.csv"

def bronze_table(entity_name: str) -> str:
    return f"bronze_{entity_name}"

def silver_table(entity_name: str) -> str:
    return f"silver_{entity_name}"

log_info("Expected entities: " + ", ".join(ENTITIES))

## Source readiness check

This cell attempts to read each source file. Missing files should be uploaded from the repository sample-data folder into the Lakehouse source folder before running ingestion.

In [ ]:
readiness_rows = []

for entity in ENTITIES:
    path = source_file(entity)
    try:
        preview_df = spark.read.format("csv").option("header", "true").option("inferSchema", "false").load(path)
        preview_df.limit(1).count()
        readiness_rows.append({
            "entity_name": entity,
            "source_path": path,
            "status": "READY",
            "column_count": len(preview_df.columns),
            "columns": ", ".join(preview_df.columns),
            "message": "Source file can be read",
        })
    except Exception as exc:
        readiness_rows.append({
            "entity_name": entity,
            "source_path": path,
            "status": "MISSING_OR_UNREADABLE",
            "column_count": 0,
            "columns": "",
            "message": str(exc)[:500],
        })

source_readiness_df = spark.createDataFrame(readiness_rows)
display(source_readiness_df.orderBy("entity_name"))

## Reusable helpers

In [ ]:
def write_delta_table(df: DataFrame, table_name: str, mode: str = "overwrite") -> int:
    row_count = df.count()
    df.write.format("delta").mode(mode).option("overwriteSchema", "true").saveAsTable(table_name)
    log_info(f"Wrote {table_name} with {row_count} rows")
    return row_count

def table_exists(table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(table_name)
    except Exception:
        try:
            spark.table(table_name).limit(1).count()
            return True
        except Exception:
            return False

def log_table_count(table_name: str) -> int:
    if not table_exists(table_name):
        log_warning(f"Table does not exist: {table_name}")
        return 0
    count_value = spark.table(table_name).count()
    log_info(f"{table_name}: {count_value} rows")
    return count_value

def display_table_counts(table_names: Iterable[str]) -> DataFrame:
    rows = []
    for table_name in table_names:
        exists = table_exists(table_name)
        rows.append({"table_name": table_name, "exists": exists, "row_count": log_table_count(table_name) if exists else 0})
    result_df = spark.createDataFrame(rows)
    display(result_df)
    return result_df

display_table_counts([bronze_table(entity) for entity in ENTITIES])

## Next steps

1. Upload the CSV files if the readiness check shows missing files.
2. Run 01_bronze_ingestion.ipynb to create raw Delta tables.
3. Run 02_silver_transformation.ipynb to clean and validate the source data.